In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

The module `RSet`, which is loaded by `setup.ml`, provides an implementation of mathematical sets that can be nested.

# Murder in the Family
---
We all know *Alice* and *Bob* from Cryptography.  However, little did
we know about their difficult family relationships, which might be the reason for
their secrecy.

Unfortunately, things have escalated recently:
Murder occurred one evening in the home of their father and mother.  Alice and Bob are the only children of their parents. One member of the family murdered another member, the third member witnessed the crime, and the fourth member was an accessory after the fact.  Furthermore, we know the following:

1. The accessory and the witness were of opposite sex.
2. The oldest member and the witness were of opposite sex.
3. The youngest member and the victim were of opposite sex.
4. The accessory was older than the victim.
5. The father was the oldest member.
6. The murderer was not the youngest member.

**Question:** Which of the four—father, mother, Alice, or Bob—was the murderer?

We use the following variables:
* `FM`, `MM`, `AM`, `BM` are true if the father, the mother, Alice, or Bob is the murderer.
* `FW`, `MW`, `AW`, `BW` are true if the father, the mother, Alice, or Bob is the witness.
* `FV`, `MV`, `AV`, `BV` are true if the father, the mother, Alice, or Bob is the victim.
* `FA`, `MA`, `AA`, `BA` are true if the father, the mother, Alice, or Bob is the accessory.
* `AY` is true if Alice is younger than Bob.

In <samp>OCaml</samp>, the names of variables have to start with a lower case letter.  Therefore, we store the set of all propositional variables in the variable `vars`.

In [ ]:
let vars = RSet.of_list [
  "FM"; "MM"; "AM"; "BM";
  "FW"; "MW"; "AW"; "BW";
  "FV"; "MV"; "AV"; "BV";
  "FA"; "MA"; "AA"; "BA";
  "AY"
]

We first construct helper constraints to ensure each person has exactly one role, and each role is assigned to exactly one person.

The function `at_least_one` takes one argument:
- `v` is a set of variables.

It returns a propositional formula, written as a string, that is true if and only if at least one variable from the set `v` is true.  `String.concat sep l` joins the strings of the list `l` and puts the separator `sep` between them.

In [ ]:
let at_least_one (v : string RSet.t) : string =
  String.concat " ∨ " (RSet.to_list v)

In [ ]:
at_least_one (RSet.of_list ["a"; "b"; "c"])

The function `at_most_one` takes one argument:
- `s` is a set of variables.

It returns a propositional formula, written as a string, that is true if and only if at most one variable from the set `s` is true.  For every pair of different variables $p$ and $q$ from `s`, the formula states that $p$ and $q$ are not both true.

In [ ]:
let at_most_one (s : string RSet.t) : string =
  let p = RSet.flat_map
            (fun p -> RSet.filter_map
                        (fun q -> if p <> q then Some (Printf.sprintf "¬(%s ∧ %s)" p q) else None)
                        s)
            s
  in
  String.concat " ∧ " (RSet.to_list p)

In [ ]:
at_most_one (RSet.of_list ["a"; "b"; "c"])

The function `exactly_one` takes one argument:
- `v` is a set of variables.

It returns a propositional formula, written as a string, that is true if and only if exactly one variable from the set `v` is true.  Replace the expression `failwith "your code here"` with your implementation.  The function `failwith` raises an exception.  As it can be used in place of an expression of any type, it serves as a placeholder in the cells that you have to complete.

In [ ]:
let exactly_one (v : string RSet.t) : string =
  failwith "your code here"

In [ ]:
exactly_one (RSet.of_list ["a"; "b"; "c"])

The set `formulas` contains the formulas stating that every person has exactly one role and that every role is assigned to exactly one person.

In [ ]:
let formulas = RSet.of_list [
  exactly_one (RSet.of_list ["FM"; "FW"; "FV"; "FA"]);
  exactly_one (RSet.of_list ["MM"; "MW"; "MV"; "MA"]);
  exactly_one (RSet.of_list ["AM"; "AW"; "AV"; "AA"]);
  exactly_one (RSet.of_list ["BM"; "BW"; "BV"; "BA"]);
  exactly_one (RSet.of_list ["FM"; "MM"; "AM"; "BM"]);
  exactly_one (RSet.of_list ["FW"; "MW"; "AW"; "BW"]);
  exactly_one (RSet.of_list ["FV"; "MV"; "AV"; "BV"]);
  exactly_one (RSet.of_list ["FA"; "MA"; "AA"; "BA"])
]

**Fact 1**: The accessory and the witness were of opposite sex.

In [ ]:
let f1 : string = failwith "your code here"

**Fact 2 & 5**: The oldest member and the witness were of opposite sex. The father was the oldest member.

In [ ]:
let f2 : string = failwith "your code here"

**Fact 3**: The youngest member and the victim were of opposite sex.

In [ ]:
let f3a : string = failwith "your code here"
let f3b : string = failwith "your code here"

**Fact 4**: The accessory was older than the victim.
Father > Mother > Older Child > Younger Child.

In [ ]:
(* your code here *)

**Fact 6**: The murderer was not the youngest member.

In [ ]:
(* your code here *)

In [ ]:
let formulas = RSet.union formulas (RSet.of_list [f1; f2; f3a; f3b; failwith "your code here"])

In [ ]:
import_notebook "Propositional-Logic-Parser.ipynb"

In [ ]:
let gs = List.map parse (RSet.to_list formulas)

The function `evaluate` takes two arguments:
- `f` is a propositional formula,
- `i` is a propositional variable assignment, i.e. the set of those variables that are true.

It evaluates `f` using the assignment `i`.

In [ ]:
let rec evaluate (f : formula) (i : string RSet.t) : bool =
  match f with
  | Var p      -> RSet.mem p i
  | Verum      -> true
  | Falsum     -> false
  | Not g      -> not (evaluate g i)
  | And (g, h) -> evaluate g i && evaluate h i
  | Or  (g, h) -> evaluate g i || evaluate h i
  | Imp (g, h) -> evaluate g i <= evaluate h i
  | Iff (g, h) -> evaluate g i =  evaluate h i

The function `all_true` takes two arguments:
- `gs` is a list of propositional formulas,
- `i` is a propositional variable assignment.

It returns `true` if and only if all formulas from `gs` are `true` given the variable assignment `i`.

In [ ]:
let all_true (gs : formula list) (i : string RSet.t) : bool =
  List.for_all (fun f -> evaluate f i) gs

We use the function `RSet.powerset` to compute all possible variable assignments and keep those that make all formulas true.  As there are 17 variables, there are $2^{17} = 131072$ assignments to check.

In [ ]:
let valid_assignments = time "search" (fun () -> RSet.filter (all_true gs) (RSet.powerset vars))

Running this model limits the valid assignments to a single solution.